# 3. Daten darstellen

Gruppe 05 (SC02_Gruppe_05): Dominik Erbsland, Joel Liechti. ZHAW, CAS Information Engineering, Scripting.

Wir lesen die Kennzahlen aus Notebook 2 in **pandas DataFrames** und zeichnen die Diagramme direkt mit `.plot()` von pandas. Zuerst ein Überblick, am Schluss die Karte (Heatmap).

In [ ]:
from pathlib import Path

import folium
import matplotlib.pyplot as plt
import pandas as pd
from folium.plugins import HeatMap

PROCESSED = Path("../data/processed")

## 3.1 Überblick: Wer, wann, warum?

Ein Bild mit vier Diagrammen fasst die wichtigsten Ergebnisse zusammen.

In [ ]:
age_gender = pd.read_csv(PROCESSED / "faelle_alter_geschlecht.csv", index_col=0)
weekday_hour = pd.read_csv(PROCESSED / "faelle_wochentag_stunde.csv", index_col=0)
reasons = pd.read_csv(PROCESSED / "faelle_grund.csv", index_col=0)["count"]
companies = pd.read_csv(PROCESSED / "faelle_unternehmen.csv", index_col=0)["count"]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# "andere (?)" hat nur 24 Fälle und wäre im Diagramm nicht sichtbar
age_gender[["männlich (?)", "weiblich (?)"]].plot(kind="bar", ax=axes[0, 0], rot=0)
axes[0, 0].set_title("Altersgruppe und Geschlecht")
axes[0, 0].set_xlabel("")
axes[0, 0].set_ylabel("Anzahl Fälle")

weekday_hour.sum(axis=0).plot(kind="bar", ax=axes[0, 1], rot=0)
axes[0, 1].set_title("Tageszeit")
axes[0, 1].set_xlabel("Stunde")

reasons.sort_values().plot(kind="barh", ax=axes[1, 0])
axes[1, 0].set_title("Grund")
axes[1, 0].set_ylabel("")

companies.head(10).sort_values().plot(kind="barh", ax=axes[1, 1])
axes[1, 1].set_title("Top 10 Verkehrsunternehmen")
axes[1, 1].set_ylabel("")

fig.suptitle("Schwarzfahrer im Schweizer ÖV", fontsize=16)
plt.tight_layout()
plt.show()

## 3.2 Wann wird am meisten erwischt?

Jedes Feld ist eine Kombination aus Wochentag und Stunde. Je dunkler, desto mehr Fälle.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.5))
image = ax.imshow(weekday_hour, cmap="YlOrRd", aspect="auto")
ax.set_yticks(range(7))
ax.set_yticklabels(weekday_hour.index)
ax.set_xticks(range(24))
ax.set_xticklabels(weekday_hour.columns)
ax.set_xlabel("Stunde")
fig.colorbar(image, label="Anzahl Fälle")
plt.show()

## 3.3 Jahrgänge

In [ ]:
years = pd.read_csv(PROCESSED / "faelle_jahrgang.csv", index_col=0)["count"]

ax = years.plot(kind="bar", figsize=(12, 4), width=0.9)
ax.set_xlabel("Jahrgang")
ax.set_ylabel("Anzahl Fälle")
# Nur jeden fünften Jahrgang beschriften, sonst ist die Achse unleserlich
ax.set_xticks(range(0, len(years), 5))
ax.set_xticklabels(years.index[::5], rotation=0)
plt.show()

## 3.4 Karte (Heatmap)

Die Karte zeigt, an welchen Starthaltestellen die meisten Fälle erfasst wurden. Hineinzoomen zeigt einzelne Orte.

In [ ]:
stations = pd.read_csv(PROCESSED / "faelle_haltestelle.csv")

# Mittelpunkt der Schweiz, Kartenhintergrund von OpenStreetMap
m = folium.Map(location=[46.8, 8.2], zoom_start=8, tiles="OpenStreetMap")

# Pro Haltestelle: Breitengrad, Längengrad und Anzahl Fälle als Gewicht
points = stations[["lat_from", "lon_from", "faelle"]].values.tolist()
HeatMap(points, radius=12).add_to(m)
m